# Notebook 03 — Category Landscape Analysis

**Amazon Market Intelligence**  
**Questions answered:** Q1 (Where should I sell?), Q2 (Is my category growing or dying?), Q3 (How competitive is my space?)  
**Tool mode:** Category Scout  
**Gold tables:** `gold_subcategory_landscape` (248), `gold_main_category_landscape` (50), `gold_temporal_trends` (8,721)

---

## 0 — Setup

In [2]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'amazon_intelligence.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)

CHARTS_DIR = 'charts/03_category_landscape'
os.makedirs(CHARTS_DIR, exist_ok=True)

TEMPLATE = 'plotly_white'
COLOR_SEQ = px.colors.qualitative.Set2

def save_chart(fig, name):
    """Save chart as interactive HTML and static PNG."""
    fig.write_html(f'{CHARTS_DIR}/{name}.html')
    try:
        fig.write_image(f'{CHARTS_DIR}/{name}.png', width=1200, height=700, scale=2)
    except Exception:
        pass

print(f'Connected to: {DB_PATH}')
print(f'Charts → {CHARTS_DIR}/')

Connected to: c:\Users\thinkpad\Desktop\amazon-market-intelligence\data\amazon_intelligence.duckdb
Charts → charts/03_category_landscape/


## 1 — Schema Discovery

In [3]:
for table in ['gold_subcategory_landscape', 'gold_main_category_landscape', 'gold_temporal_trends']:
    print(f'\n=== {table} ===')
    print(con.sql(f'DESCRIBE {table}').df().to_string())
    print(f'Rows: {con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0]:,}')


=== gold_subcategory_landscape ===
                column_name column_type null   key default extra
0               subcategory     VARCHAR  YES  None    None  None
1             product_count      BIGINT  YES  None    None  None
2                 avg_price      DOUBLE  YES  None    None  None
3              median_price      DOUBLE  YES  None    None  None
4                avg_rating      DOUBLE  YES  None    None  None
5               avg_reviews      DOUBLE  YES  None    None  None
6          total_units_sold     HUGEINT  YES  None    None  None
7             total_revenue      DOUBLE  YES  None    None  None
8   avg_revenue_per_product      DOUBLE  YES  None    None  None
9          pct_best_sellers      DOUBLE  YES  None    None  None
10               pct_active      DOUBLE  YES  None    None  None
11         pct_zero_reviews      DOUBLE  YES  None    None  None
12         avg_discount_pct      DOUBLE  YES  None    None  None
13           pct_with_brand      DOUBLE  YES  None    

## 2 — Load Gold Tables

In [4]:
df_sub = con.sql('SELECT * FROM gold_subcategory_landscape').df()
df_main = con.sql('SELECT * FROM gold_main_category_landscape').df()
df_trends = con.sql('SELECT * FROM gold_temporal_trends').df()

print(f'Subcategory landscape: {df_sub.shape}')
print(f'Main category landscape: {df_main.shape}')
print(f'Temporal trends: {df_trends.shape}')

df_sub.head(3)

Subcategory landscape: (248, 17)
Main category landscape: (50, 18)
Temporal trends: (8721, 14)


,subcategory,product_count,avg_price,median_price,avg_rating,avg_reviews,total_units_sold,total_revenue,avg_revenue_per_product,pct_best_sellers,pct_active,pct_zero_reviews,avg_discount_pct,pct_with_brand,pct_with_features,pct_with_description,pct_with_store
0,Kitchen & Dining,4882,26.42,15.99,4.54,0.0,10432300.0,267189588.0,54729.53,5.7,99.1,100.0,23.8,21.0,24.7,11.3,25.0
1,Hair Care Products,8669,20.93,14.50,4.41,0.0,8012850.0,152940697.5,17642.25,0.5,99.7,100.0,23.0,26.3,31.3,18.7,37.8
2,Home Storage & Organization,15437,52.75,23.49,4.26,0.0,5356200.0,138604708.5,8978.73,0.8,58.2,100.0,21.1,21.9,25.1,14.4,25.8


In [5]:
df_sub.columns.tolist()

['subcategory',
 'product_count',
 'avg_price',
 'median_price',
 'avg_rating',
 'avg_reviews',
 'total_units_sold',
 'total_revenue',
 'avg_revenue_per_product',
 'pct_best_sellers',
 'pct_active',
 'pct_zero_reviews',
 'avg_discount_pct',
 'pct_with_brand',
 'pct_with_features',
 'pct_with_description',
 'pct_with_store']

In [6]:
df_main.columns.tolist()

['main_category',
 'ecosystem_products',
 'store_count',
 'brand_count',
 'kaggle_products',
 'kaggle_coverage_pct',
 'total_units_sold',
 'total_revenue',
 'avg_revenue_per_product',
 'avg_price',
 'avg_rating',
 'pct_active',
 'products_per_store',
 'revenue_per_store',
 'pct_with_features',
 'pct_with_description',
 'pct_with_brand',
 'pct_with_store']

In [7]:
df_trends.columns.tolist()

['source_category',
 'review_year',
 'review_month',
 'review_count',
 'avg_rating',
 'median_rating',
 'pct_verified',
 'pct_with_images',
 'avg_helpful_votes',
 'avg_text_length',
 'negative_reviews',
 'neutral_reviews',
 'positive_reviews',
 'pct_negative']

---

## 3 — The Marketplace at a Glance

### 3.1 — Top 20 Subcategories by Revenue

**Q1: Where should I sell?** — Follow the money.

In [8]:
REVENUE_COL = 'total_revenue'  
PRODUCTS_COL = 'product_count'  
SUBCAT_COL = 'subcategory'      

top20_rev = df_sub.nlargest(20, REVENUE_COL)

fig = px.bar(
    top20_rev,
    x=REVENUE_COL,
    y=SUBCAT_COL,
    orientation='h',
    title='Top 20 Subcategories by Estimated Revenue',
    labels={REVENUE_COL: 'Estimated Revenue ($)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#2196F3']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.update_traces(texttemplate='$%{x:,.0f}', textposition='outside')
save_chart(fig, '01_top20_revenue')
fig.show()

### 3.2 — Revenue per Product (Demand Density)

Raw revenue is misleading — a category with $100M spread across 50K products is very different from $100M across 500.  
**Revenue per product** shows where demand concentrates.

In [9]:
df_sub['revenue_per_product'] = df_sub[REVENUE_COL] / df_sub[PRODUCTS_COL].replace(0, np.nan)

top20_density = df_sub.nlargest(20, 'revenue_per_product')

fig = px.bar(
    top20_density,
    x='revenue_per_product',
    y=SUBCAT_COL,
    orientation='h',
    title='Top 20 Subcategories by Revenue per Product (Demand Density)',
    labels={'revenue_per_product': 'Revenue per Product ($)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#FF9800']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.update_traces(texttemplate='$%{x:,.0f}', textposition='outside')
save_chart(fig, '02_revenue_per_product')
fig.show()

### 3.3 — The Opportunity Quadrant

**X-axis:** Revenue per product (how much is being spent per listing)  
**Y-axis:** Activity rate (what % of listings actually sell)  
**Size:** Total product count (competition level)  

The sweet spot: **top-right, small bubble** = high demand per product, high activity, low competition.

In [10]:
ACTIVE_COL = 'pct_active'
fig = px.scatter(
    df_sub,
    x='revenue_per_product',
    y=ACTIVE_COL,
    size=PRODUCTS_COL,
    hover_name=SUBCAT_COL,
    title='Category Opportunity Quadrant — Where Demand Meets Activity',
    labels={
        'revenue_per_product': 'Revenue per Product ($)',
        ACTIVE_COL: 'Activity Rate (%)',
        PRODUCTS_COL: 'Product Count'
    },
    template=TEMPLATE,
    color_discrete_sequence=['#4CAF50'],
    size_max=50
)

med_rev = df_sub['revenue_per_product'].median()
med_act = df_sub[ACTIVE_COL].median()
fig.add_hline(y=med_act, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=med_rev, line_dash='dash', line_color='gray', opacity=0.5)

fig.add_annotation(x=med_rev*3, y=med_act*1.3, text='🎯 Sweet Spot', showarrow=False,
                   font=dict(size=14, color='green'))
fig.add_annotation(x=med_rev*0.2, y=med_act*0.5, text='⚠️ Graveyard', showarrow=False,
                   font=dict(size=14, color='red'))

fig.update_layout(height=700)
save_chart(fig, '03_opportunity_quadrant')
fig.show()

### 3.4 — Activity Rate Distribution

**Q2 context:** How alive is the marketplace? What's the typical activity rate?

In [11]:
fig = px.histogram(
    df_sub,
    x=ACTIVE_COL,
    nbins=30,
    title='Distribution of Activity Rates Across 248 Subcategories',
    labels={ACTIVE_COL: 'Activity Rate (% of products with sales > 0)'},
    template=TEMPLATE,
    color_discrete_sequence=['#9C27B0']
)
fig.add_vline(x=df_sub[ACTIVE_COL].median(), line_dash='dash', line_color='red',
              annotation_text=f'Median: {df_sub[ACTIVE_COL].median():.1f}%')
fig.update_layout(height=400)
save_chart(fig, '04_activity_distribution')
fig.show()

### 3.5 — Ghost Marketplace: Bottom 20 by Activity

Categories where most listings are dead weight.

In [12]:
bottom20_activity = df_sub.nsmallest(20, ACTIVE_COL)

fig = px.bar(
    bottom20_activity,
    x=ACTIVE_COL,
    y=SUBCAT_COL,
    orientation='h',
    title='Bottom 20 Subcategories by Activity Rate — The Ghost Categories',
    labels={ACTIVE_COL: 'Activity Rate (%)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#F44336']
)
fig.update_layout(yaxis={'categoryorder': 'total descending'}, height=600)
fig.update_traces(texttemplate='%{x:.1f}%', textposition='outside')
save_chart(fig, '05_ghost_categories')
fig.show()

---

## 4 — Main Category Ecosystem (35M products)

Zoom out: the full Amazon landscape from McAuley metadata. This includes products with no sales data — the total catalog.

In [13]:
df_main.head()

,main_category,ecosystem_products,store_count,brand_count,kaggle_products,kaggle_coverage_pct,total_units_sold,total_revenue,avg_revenue_per_product,avg_price,avg_rating,pct_active,products_per_store,revenue_per_store,pct_with_features,pct_with_description,pct_with_brand,pct_with_store
0,Amazon Home,4369831,429032,375613,39866,0.91,9063850.0,224949780.0,5642.65,30.39,4.43,61.8,10.2,524.32,76.9,57.3,76.6,100.0
1,All Beauty,953702,121631,99074,17832,1.87,10497050.0,170235981.0,9546.66,17.46,4.39,79.7,7.8,1399.61,68.0,53.0,70.8,100.0
2,Health & Personal Care,658634,138452,102845,15752,2.39,7647400.0,155792797.0,9890.35,23.87,4.36,70.0,4.8,1125.25,73.2,56.3,66.8,100.0
3,Toys & Games,899219,98751,54884,54112,6.02,7242500.0,149941626.5,2770.95,24.97,4.50,53.9,9.1,1518.38,80.0,67.2,37.0,100.0
4,Tools & Home Improvement,1664632,216734,184547,51370,3.09,3881450.0,112930723.5,2198.38,35.26,4.40,25.8,7.7,521.06,79.6,58.0,72.7,100.0


In [14]:
MAINCAT_COL = 'main_category'
MAIN_PRODUCTS_COL = 'ecosystem_products'

fig = px.bar(
    df_main.nlargest(25, MAIN_PRODUCTS_COL),
    x=MAIN_PRODUCTS_COL,
    y=MAINCAT_COL,
    orientation='h',
    title='Amazon Full Ecosystem — Top 25 Main Categories by Catalog Size (35M products)',
    labels={MAIN_PRODUCTS_COL: 'Total Products', MAINCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#00BCD4']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700)
fig.update_traces(texttemplate='%{x:,.0f}', textposition='outside')
save_chart(fig, '06_main_category_size')
fig.show()

### 4.1 — Catalog Size vs Brand Presence

Do bigger categories have stronger brand identity? Or is brand a niche advantage?

In [15]:
print(df_main.columns.tolist())
print()
df_main.describe()

['main_category', 'ecosystem_products', 'store_count', 'brand_count', 'kaggle_products', 'kaggle_coverage_pct', 'total_units_sold', 'total_revenue', 'avg_revenue_per_product', 'avg_price', 'avg_rating', 'pct_active', 'products_per_store', 'revenue_per_store', 'pct_with_features', 'pct_with_description', 'pct_with_brand', 'pct_with_store']



,ecosystem_products,store_count,brand_count,kaggle_products,kaggle_coverage_pct,total_units_sold,total_revenue,avg_revenue_per_product,avg_price,avg_rating,pct_active,products_per_store,revenue_per_store,pct_with_features,pct_with_description,pct_with_brand,pct_with_store
count,5.000000e+01,5.000000e+01,50.00000,40.0,40.000000,4.000000e+01,4.000000e+01,40.000000,40.000000,40.000000,40.000000,50.00000,40.000000,50.000000,50.000000,50.00000,50.000000
mean,6.526510e+05,1.162207e+05,32453.26000,9540.2,2.078750,1.395255e+06,3.174666e+07,4727.331000,52.861000,4.249250,28.642500,6993.08800,3496.419500,68.940000,55.878000,38.61400,99.998000
std,1.329537e+06,3.123764e+05,63499.76496,14573.192569,2.496441,2.647024e+06,5.562141e+07,9745.299151,56.058713,0.716914,29.973072,49371.19721,14654.434209,27.842861,26.812914,32.78037,0.014142
min,1.000000e+00,1.000000e+00,0.00000,2.0,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,1.00000,0.000000,0.000000,0.000000,0.00000,99.900000
25%,1.046725e+04,7.985000e+02,15.25000,53.75,0.287500,7.125000e+02,1.856450e+04,41.295000,22.325000,4.267500,0.675000,2.77500,0.235000,68.125000,45.225000,0.82500,100.000000
50%,1.397275e+05,2.340200e+04,1009.50000,1472.5,1.185000,3.157500e+04,6.986118e+05,1431.275000,34.495000,4.370000,22.950000,6.65000,124.000000,76.700000,58.900000,40.90000,100.000000
75%,6.477545e+05,9.736625e+04,46036.00000,13663.75,3.147500,1.364138e+06,3.275022e+07,5058.460000,61.285000,4.465000,47.475000,8.95000,718.247500,83.600000,71.475000,66.15000,100.000000
max,7.374342e+06,2.040285e+06,375613.00000,54112.0,12.050000,1.049705e+07,2.249498e+08,51856.880000,285.450000,4.740000,100.000000,349118.00000,84436.280000,100.000000,100.000000,100.00000,100.000000


---

## 5 — Temporal Trends (Q2: Growing or Dying?)

The temporal trends table has year-level review data by category. Review volume is a proxy for market activity.

In [16]:
df_trends.head()

,source_category,review_year,review_month,review_count,avg_rating,median_rating,pct_verified,pct_with_images,avg_helpful_votes,avg_text_length,negative_reviews,neutral_reviews,positive_reviews,pct_negative
0,All_Beauty,2000,11,1,5.0,5.0,0.0,0.0,10.0,239.0,0.0,0.0,1.0,0.0
1,All_Beauty,2001,1,2,3.5,3.5,0.0,0.0,22.0,1471.0,1.0,0.0,1.0,50.0
2,All_Beauty,2001,3,2,5.0,5.0,0.0,0.0,5.5,1232.5,0.0,0.0,2.0,0.0
3,All_Beauty,2001,4,2,4.0,4.0,0.0,0.0,7.0,365.0,0.0,0.0,2.0,0.0
4,All_Beauty,2001,9,1,5.0,5.0,0.0,0.0,3.0,595.0,0.0,0.0,1.0,0.0


In [17]:

YEAR_COL = 'review_year'
TREND_CAT_COL = 'source_category'
REVIEW_VOL_COL = 'review_count'
AVG_RATING_COL = 'avg_rating'

df_t = (
    df_trends[df_trends[YEAR_COL].between(2018, 2022)]
    .groupby([TREND_CAT_COL, YEAR_COL])
    .agg({REVIEW_VOL_COL: 'sum', AVG_RATING_COL: 'mean'})
    .reset_index()
)

print(f'Year range in data: {df_trends[YEAR_COL].min()} — {df_trends[YEAR_COL].max()}')
print(f'Categories: {df_trends[TREND_CAT_COL].nunique()}')
print(f'Filtered rows (2018-2022): {len(df_t):,}')

Year range in data: 2000 — 2023
Categories: 33
Filtered rows (2018-2022): 165


### 5.1 — Overall Review Volume Trend

Is Amazon review activity growing or plateauing?

In [18]:
yearly_total = df_t.groupby(YEAR_COL)[REVIEW_VOL_COL].sum().reset_index()

fig = px.bar(
    yearly_total,
    x=YEAR_COL,
    y=REVIEW_VOL_COL,
    title='Total Review Volume by Year (All Categories)',
    labels={REVIEW_VOL_COL: 'Reviews', YEAR_COL: 'Year'},
    template=TEMPLATE,
    color_discrete_sequence=['#3F51B5'],
    text_auto=True
)
fig.update_layout(height=400)
save_chart(fig, '07_review_volume_trend')
fig.show()

### 5.2 — Average Rating Decline

Finding #35: Ratings are declining 4.28 → 4.02 (2019-2022). Let's visualize the trajectory.

In [19]:
yearly_rating = (
    df_t.groupby(YEAR_COL)
    .apply(lambda g: np.average(g[AVG_RATING_COL], weights=g[REVIEW_VOL_COL]), include_groups=False)
    .reset_index(name='weighted_avg_rating')
)

fig = px.line(
    yearly_rating,
    x=YEAR_COL,
    y='weighted_avg_rating',
    title='Average Rating Decline Across Amazon (2018–2022)',
    labels={'weighted_avg_rating': 'Weighted Avg Rating', YEAR_COL: 'Year'},
    template=TEMPLATE,
    markers=True
)
fig.update_traces(line=dict(width=3, color='#E91E63'))
fig.update_yaxes(range=[3.8, 4.5])
fig.update_layout(height=400)
save_chart(fig, '08_rating_decline')
fig.show()

### 5.3 — Category Growth Heatmap

Which categories are gaining review momentum? Which are fading?  
Show year-over-year % change in review volume.

In [20]:
pivot = df_t.pivot_table(
    index=TREND_CAT_COL,
    columns=YEAR_COL,
    values=REVIEW_VOL_COL,
    aggfunc='sum'
).fillna(0)

growth = pivot.pct_change(axis=1) * 100
growth = growth.drop(columns=growth.columns[0])  # first year has no baseline

last_year = growth.columns[-1]
growth = growth.sort_values(last_year, ascending=True)

fig = px.imshow(
    growth,
    title='Category Growth Heatmap — YoY Review Volume Change (%)',
    labels=dict(x='Year', y='Category', color='YoY Change (%)'),
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    aspect='auto'
)
fig.update_layout(height=900)
save_chart(fig, '09_growth_heatmap')
fig.show()

### 5.4 — Top Growers vs Top Decliners

Rank categories by compound growth rate (2018→2022).

In [21]:
start_year = pivot.columns[0]
end_year = pivot.columns[-1]
years = end_year - start_year

cagr = pd.DataFrame({
    TREND_CAT_COL: pivot.index,
    'start_vol': pivot[start_year].values,
    'end_vol': pivot[end_year].values
})
cagr = cagr[cagr['start_vol'] > 0]  
cagr['cagr_pct'] = ((cagr['end_vol'] / cagr['start_vol']) ** (1 / years) - 1) * 100

top_growers = cagr.nlargest(10, 'cagr_pct')
top_decliners = cagr.nsmallest(10, 'cagr_pct')
extreme = pd.concat([top_growers, top_decliners]).sort_values('cagr_pct')

colors = ['#F44336' if x < 0 else '#4CAF50' for x in extreme['cagr_pct']]

fig = go.Figure(go.Bar(
    x=extreme['cagr_pct'],
    y=extreme[TREND_CAT_COL],
    orientation='h',
    marker_color=colors,
    text=[f'{x:+.1f}%' for x in extreme['cagr_pct']],
    textposition='outside'
))
fig.update_layout(
    title=f'Category Growth Champions & Decliners (CAGR {start_year}–{end_year})',
    xaxis_title='Compound Annual Growth Rate (%)',
    template=TEMPLATE,
    height=600
)
save_chart(fig, '10_growers_vs_decliners')
fig.show()

---

## 6 — Competition Intensity (Q3: How competitive is my space?)

Competition isn't just product count. It's the combination of:
- How many products exist (supply)
- How few are actually selling (activity rate)
- How concentrated revenue is (revenue per product)

Low activity + high product count = **oversaturated graveyard**.

In [22]:
df_sub['competition_score'] = df_sub[PRODUCTS_COL] / df_sub[ACTIVE_COL].replace(0, np.nan)

most_competitive = df_sub.nlargest(20, 'competition_score')

fig = px.bar(
    most_competitive,
    x='competition_score',
    y=SUBCAT_COL,
    orientation='h',
    title='Most Oversaturated Categories (High Product Count ÷ Low Activity)',
    labels={'competition_score': 'Competition Score (higher = harder)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#FF5722']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
save_chart(fig, '11_competition_intensity')
fig.show()

### 6.1 — Hidden Gems: High Demand, Low Competition

Categories with high revenue per product but relatively few competitors.

In [23]:
df_sub['opportunity_score'] = (
    df_sub['revenue_per_product'] * df_sub[ACTIVE_COL]
) / df_sub[PRODUCTS_COL].replace(0, np.nan)

gems = df_sub.nlargest(15, 'opportunity_score')

fig = px.scatter(
    gems,
    x=PRODUCTS_COL,
    y='revenue_per_product',
    size=ACTIVE_COL,
    hover_name=SUBCAT_COL,
    text=SUBCAT_COL,
    title='Hidden Gems — High Revenue per Product, Low Competition',
    labels={
        PRODUCTS_COL: 'Total Products (lower = less competition)',
        'revenue_per_product': 'Revenue per Product ($)',
        ACTIVE_COL: 'Activity Rate'
    },
    template=TEMPLATE,
    color_discrete_sequence=['#FF9800'],
    size_max=40
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=600)
save_chart(fig, '12_hidden_gems')
fig.show()

---

## 7 — Category Typology

Classify every subcategory into one of 4 strategic types based on data:

| Type | Revenue/Product | Activity | What it means |
|------|----------------|----------|---------------|
| 🌟 Star | High | High | Money is here, things sell |
| 💀 Graveyard | Low | Low | Dead listings, nobody buys |
| 🏭 Volume Play | Low | High | Things sell but margins are thin |
| 💎 Niche Premium | High | Low | Few sell but those that do earn big |

In [24]:
med_rev = df_sub['revenue_per_product'].median()
med_act = df_sub[ACTIVE_COL].median()

def classify(row):
    high_rev = row['revenue_per_product'] >= med_rev
    high_act = row[ACTIVE_COL] >= med_act
    if high_rev and high_act:
        return '🌟 Star'
    elif high_rev and not high_act:
        return '💎 Niche Premium'
    elif not high_rev and high_act:
        return '🏭 Volume Play'
    else:
        return '💀 Graveyard'

df_sub['category_type'] = df_sub.apply(classify, axis=1)

type_counts = df_sub['category_type'].value_counts()
print('Category Typology Distribution:')
for t, c in type_counts.items():
    print(f'  {t}: {c} subcategories')

# Scatter colored by type
fig = px.scatter(
    df_sub,
    x='revenue_per_product',
    y=ACTIVE_COL,
    color='category_type',
    hover_name=SUBCAT_COL,
    title='Category Typology — 248 Subcategories Classified',
    labels={
        'revenue_per_product': 'Revenue per Product ($)',
        ACTIVE_COL: 'Activity Rate (%)',
        'category_type': 'Type'
    },
    template=TEMPLATE,
    color_discrete_map={
        '🌟 Star': '#4CAF50',
        '💎 Niche Premium': '#FF9800',
        '🏭 Volume Play': '#2196F3',
        '💀 Graveyard': '#9E9E9E'
    }
)
fig.add_hline(y=med_act, line_dash='dash', line_color='gray', opacity=0.4)
fig.add_vline(x=med_rev, line_dash='dash', line_color='gray', opacity=0.4)
fig.update_layout(height=700)
save_chart(fig, '13_category_typology')
fig.show()

Category Typology Distribution:
  🌟 Star: 109 subcategories
  💀 Graveyard: 109 subcategories
  💎 Niche Premium: 15 subcategories
  🏭 Volume Play: 15 subcategories


In [25]:
rev_by_type = df_sub.groupby('category_type')[REVENUE_COL].sum().reset_index()
rev_by_type['pct'] = (rev_by_type[REVENUE_COL] / rev_by_type[REVENUE_COL].sum() * 100).round(1)

fig = px.pie(
    rev_by_type,
    values=REVENUE_COL,
    names='category_type',
    title='Revenue Share by Category Type',
    template=TEMPLATE,
    color='category_type',
    color_discrete_map={
        '🌟 Star': '#4CAF50',
        '💎 Niche Premium': '#FF9800',
        '🏭 Volume Play': '#2196F3',
        '💀 Graveyard': '#9E9E9E'
    }
)
fig.update_traces(textinfo='label+percent', textfont_size=12)
fig.update_layout(height=450)
save_chart(fig, '14_revenue_by_type')
fig.show()

---

## 8 — Key Findings & Story Takeaways

Summarize what a Category Scout user would care about.

In [26]:
print('=' * 60)
print('CATEGORY LANDSCAPE — KEY FINDINGS')
print('=' * 60)

top1 = df_sub.nlargest(1, REVENUE_COL).iloc[0]
print(f'\n1. BIGGEST CATEGORY: {top1[SUBCAT_COL]}')
print(f'   Revenue: ${top1[REVENUE_COL]:,.0f} | Products: {top1[PRODUCTS_COL]:,.0f} | Activity: {top1[ACTIVE_COL]:.1f}%')

top_density = df_sub.nlargest(1, 'revenue_per_product').iloc[0]
print(f'\n2. HIGHEST DEMAND DENSITY: {top_density[SUBCAT_COL]}')
print(f'   Revenue/Product: ${top_density["revenue_per_product"]:,.0f}')

star_count = (df_sub['category_type'] == '🌟 Star').sum()
grave_count = (df_sub['category_type'] == '💀 Graveyard').sum()
print(f'\n3. TYPOLOGY: {star_count} Stars vs {grave_count} Graveyards (of 248 subcategories)')

star_rev_pct = rev_by_type[rev_by_type['category_type'] == '🌟 Star']['pct'].values
if len(star_rev_pct) > 0:
    print(f'   Stars capture {star_rev_pct[0]}% of total revenue')

print(f'\n4. RATING TREND: Declining across the marketplace')
if len(yearly_rating) >= 2:
    r_start = yearly_rating.iloc[0]['weighted_avg_rating']
    r_end = yearly_rating.iloc[-1]['weighted_avg_rating']
    print(f'   {r_start:.2f} → {r_end:.2f} ({r_start - r_end:+.2f} decline)')

print(f'\n5. TOP GROWERS: {top_growers[TREND_CAT_COL].iloc[0]} ({top_growers["cagr_pct"].iloc[0]:+.1f}% CAGR)')
print(f'   TOP DECLINERS: {top_decliners[TREND_CAT_COL].iloc[0]} ({top_decliners["cagr_pct"].iloc[0]:+.1f}% CAGR)')

print('\n' + '=' * 60)

CATEGORY LANDSCAPE — KEY FINDINGS

1. BIGGEST CATEGORY: Kitchen & Dining
   Revenue: $267,189,588 | Products: 4,882 | Activity: 99.1%

2. HIGHEST DEMAND DENSITY: Health & Household
   Revenue/Product: $129,690

3. TYPOLOGY: 109 Stars vs 109 Graveyards (of 248 subcategories)
   Stars capture 91.7% of total revenue

4. RATING TREND: Declining across the marketplace
   4.22 → 4.02 (+0.19 decline)

5. TOP GROWERS: Patio_Lawn_and_Garden (+22.1% CAGR)
   TOP DECLINERS: Magazine_Subscriptions (-23.5% CAGR)



In [27]:
con.close()
print('Done. DuckDB connection closed.')
print(f'Charts saved to: {CHARTS_DIR}/')

Done. DuckDB connection closed.
Charts saved to: charts/03_category_landscape/
